# 02 — Data Understanding
## CRISP-DM Phase 2: NHANES Raw Microdata Inventory, SEQN Auditing & Quality Check

---

### CONCEPT
This notebook inventories the raw CDC/NCHS NHANES August 2021-2023 microdata files, audits the integration key (`SEQN`), and profiles data quality before any filtering or transformation.

### WHY
Data Understanding is the CRISP-DM gate that prevents downstream errors caused by undocumented file structures, duplicate keys, unexpected missing patterns, or misunderstood variable semantics. Every decision in later phases (cohort filtering, feature engineering, model evaluation) depends on the integrity of this raw data foundation.

### INPUT
- **8 raw SAS Transport files** (`.xpt`) from CDC NHANES public release
- Located in `nhanes_feasibility_2021_2023/raw/`


In [1]:
import sys, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.sans-serif': 'DejaVu Sans',
    'figure.autolayout': True,
    'axes.edgecolor': '#333333',
    'axes.linewidth': 0.8,
    'figure.figsize': (10, 4)
})

try:
    display
except NameError:
    try:
        from IPython.display import display
    except ImportError:
        def display(*args, **kwargs):
            for a in args:
                if hasattr(a, 'to_string'):
                    print(a.to_string())
                else:
                    print(a)

# Robust path resolution
current = Path.cwd().resolve()
candidates = [current, current.parent, current.parent.parent]
PROJECT_ROOT = None
for c in candidates:
    if (c / "nhanes_feasibility_2021_2023").exists():
        PROJECT_ROOT = c
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = current

NHANES_DIR = PROJECT_ROOT / "nhanes_feasibility_2021_2023"
RAW_DIR = NHANES_DIR / "raw"
print(f"Project Root: {PROJECT_ROOT}")
print(f"Raw XPT Directory: {RAW_DIR}")
assert RAW_DIR.exists(), f"Raw directory missing: {RAW_DIR}"


Project Root: C:\Users\Felix\Documents\Skripsi
Raw XPT Directory: C:\Users\Felix\Documents\Skripsi\nhanes_feasibility_2021_2023\raw


## 1. Raw File Inventory & Integrity

### PROCESS
We read all 8 SAS Transport files, recording their dimensions, file sizes, and SHA-256 hashes for reproducibility. A SAS floating-point underflow normalization (`|x| < 1e-10 -> 0.0`) is applied because SAS Transport files occasionally encode exact numeric zeros as extremely small floating-point values (~1e-79).


In [2]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def normalize_sas_zeros(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col] = np.where((df[col].notna()) & (df[col].abs() < 1e-10), 0.0, df[col])
    return df

raw_files = [
    "DEMO_L.xpt", "BMX_L.xpt", "BPQ_L.xpt", "SMQ_L.xpt",
    "PAQ_L.xpt", "DIQ_L.xpt", "GHB_L.xpt", "GLU_L.xpt"
]

raw_manifest = []
dfs = {}

for fname in raw_files:
    fpath = RAW_DIR / fname
    assert fpath.exists(), f"Missing raw file: {fpath}"
    sha = sha256_file(fpath)
    df_raw = normalize_sas_zeros(pd.read_sas(str(fpath), format="xport"))
    dfs[fname.split(".")[0]] = df_raw
    raw_manifest.append({
        "file": fname,
        "rows": len(df_raw),
        "columns": df_raw.shape[1],
        "size_kb": round(fpath.stat().st_size / 1024, 1),
        "sha256_prefix": sha[:16] + "..."
    })

manifest_df = pd.DataFrame(raw_manifest)
print("=== NHANES August 2021-2023 Raw File Inventory ===")
display(manifest_df)
print(f"\nTotal files: {len(raw_manifest)}")
print(f"Demographic base (DEMO_L): {len(dfs['DEMO_L']):,} participants")


=== NHANES August 2021-2023 Raw File Inventory ===
         file   rows  columns  size_kb        sha256_prefix
0  DEMO_L.xpt  11933       27   2521.6  ca4374a158b493b8...
1   BMX_L.xpt   8860       22   1526.6  44440c416d9ad709...
2   BPQ_L.xpt   8501        6    400.1  0acde3af0526c375...
3   SMQ_L.xpt   9015        9    635.9  abbefebfa5585b1e...
4   PAQ_L.xpt   8153        8    399.9  de34acfed4523f10...
5   DIQ_L.xpt  11744        9    827.7  9535a023673ae869...
6   GHB_L.xpt   7199        3    169.9  67aee0353160e239...
7   GLU_L.xpt   3996        4    126.2  70f773ab37e3d848...

Total files: 8
Demographic base (DEMO_L): 11,933 participants


### INTERPRETATION
All 8 files are present and readable. `DEMO_L.xpt` contains the full survey sample (N=11,933) serving as the merge base. Laboratory files (`GHB_L`, `GLU_L`) have smaller row counts because not all participants completed phlebotomy or the fasting glucose subsample.


## 2. SEQN Integration Key Audit

### CONCEPT
`SEQN` (Respondent Sequence Number) is the unique participant identifier across all NHANES components. Before merging, we must verify:
1. `SEQN` is unique (zero duplicates) within every file
2. All component files' SEQNs are subsets of DEMO_L SEQNs
3. The left-join merge produces exactly N=11,933 rows


In [3]:
# SEQN uniqueness audit
print("=== SEQN Uniqueness Audit ===")
seqn_audit = []
for fname, df_raw in dfs.items():
    n_total = len(df_raw)
    n_unique = df_raw["SEQN"].nunique()
    has_dupes = n_total != n_unique
    seqn_audit.append({
        "file": fname,
        "total_rows": n_total,
        "unique_SEQN": n_unique,
        "duplicates": n_total - n_unique,
        "status": "DUPLICATES" if has_dupes else "UNIQUE"
    })

seqn_df = pd.DataFrame(seqn_audit)
display(seqn_df)

# Verify all component SEQNs are subsets of DEMO_L
demo_seqns = set(dfs["DEMO_L"]["SEQN"].values)
print("\n=== SEQN Subset Coverage ===")
for fname, df_raw in dfs.items():
    if fname == "DEMO_L":
        continue
    comp_seqns = set(df_raw["SEQN"].values)
    orphans = comp_seqns - demo_seqns
    coverage = len(comp_seqns & demo_seqns) / len(comp_seqns) * 100 if len(comp_seqns) > 0 else 0
    status = "PASS" if len(orphans) == 0 else f"{len(orphans)} orphans"
    print(f"  {fname}: {coverage:.1f}% coverage | {status}")


=== SEQN Uniqueness Audit ===
     file  total_rows  unique_SEQN  duplicates  status
0  DEMO_L       11933        11933           0  UNIQUE
1   BMX_L        8860         8860           0  UNIQUE
2   BPQ_L        8501         8501           0  UNIQUE
3   SMQ_L        9015         9015           0  UNIQUE
4   PAQ_L        8153         8153           0  UNIQUE
5   DIQ_L       11744        11744           0  UNIQUE
6   GHB_L        7199         7199           0  UNIQUE
7   GLU_L        3996         3996           0  UNIQUE

=== SEQN Subset Coverage ===
  BMX_L: 100.0% coverage | PASS
  BPQ_L: 100.0% coverage | PASS
  SMQ_L: 100.0% coverage | PASS
  PAQ_L: 100.0% coverage | PASS
  DIQ_L: 100.0% coverage | PASS
  GHB_L: 100.0% coverage | PASS
  GLU_L: 100.0% coverage | PASS


## 3. Participant-Level Merge & Raw Population Count

### PROCESS
All 7 component files are left-joined onto `DEMO_L` using `SEQN`, preserving the full demographic base.


In [4]:
# Left-join merge
demo = dfs["DEMO_L"].copy()
merged = demo.copy()

merge_order = ["BMX_L", "BPQ_L", "SMQ_L", "PAQ_L", "DIQ_L", "GHB_L", "GLU_L"]
for key in merge_order:
    comp = dfs[key].copy()
    overlap = (set(merged.columns) & set(comp.columns)) - {"SEQN"}
    if overlap:
        comp = comp.drop(columns=list(overlap))
    merged = merged.merge(comp, on="SEQN", how="left")

merged = normalize_sas_zeros(merged)
print(f"Merged Base: {merged.shape[0]:,} participants x {merged.shape[1]} raw variables")
assert len(merged) == 11933, f"Expected 11,933 rows, got {len(merged)}"
assert merged["SEQN"].is_unique, "Fatal: Duplicate SEQN detected!"
print("Merge integrity verified: N=11,933, zero duplicate SEQNs")


Merged Base: 11,933 participants x 81 raw variables
Merge integrity verified: N=11,933, zero duplicate SEQNs


### PREVIEW: Inspeksi Data Mikro Gabungan (`merged.head()`)

### TUJUAN
Cell di bawah ini bertujuan untuk melihat beberapa baris pertama data mentah yang telah digabung menggunakan fungsi `head()` (`merged.head()`). Melalui fungsi ini, peneliti dapat menginspeksi secara langsung struktur kolom gabungan dari 8 file komponen survei NHANES (Demografi, Pemeriksaan Fisik BMX, Kuesioner BPQ/SMQ/PAQ/DIQ, serta Laboratorium GHB/GLU) dan memeriksa nilai-nilai sampel responden.


In [5]:
# Tampilkan 5 baris pertama data mentah yang telah digabung (merged microdata)
display(merged.head())


       SEQN  SDDSRVYR  RIDSTATR  RIAGENDR  RIDAGEYR  RIDAGEMN  ...  DIQ070       WTPH2YR  LBXGH       WTSAF2YR  LBXGLU  LBDGLUSI
0  130378.0      12.0       2.0       1.0      43.0       NaN  ...     NaN  56042.129410    5.6  120025.308444   113.0      6.27
1  130379.0      12.0       2.0       1.0      66.0       NaN  ...     NaN  37435.705647    5.6       0.000000    99.0      5.50
2  130380.0      12.0       2.0       2.0      44.0       NaN  ...     1.0  85328.844519    6.2  145090.773569   156.0      8.66
3  130381.0      12.0       2.0       2.0       5.0       NaN  ...     NaN           NaN    NaN            NaN     NaN       NaN
4  130382.0      12.0       2.0       1.0       2.0       NaN  ...     NaN           NaN    NaN            NaN     NaN       NaN


## 4. Row Progression Audit: From Raw to Eligible

### CONCEPT
Before formal cohort construction (Notebook 03), we preview the participant attrition from the full NHANES sample to the eligible screening population.


In [6]:
# Row progression: Raw -> Adults -> No known DM/PreDM -> Valid HbA1c
n_total = len(merged)
n_adults = int((merged["RIDAGEYR"] >= 18).sum())
n_no_dm = int(((merged["RIDAGEYR"] >= 18) & (merged["DIQ010"] == 2)).sum())
n_cohort_e = int(((merged["RIDAGEYR"] >= 18) & (merged["DIQ010"] == 2) & (merged["DIQ160"] == 2)).sum())

valid_hba1c_mask = (merged["RIDAGEYR"] >= 18) & (merged["DIQ010"] == 2) & (merged["DIQ160"] == 2) & merged["LBXGH"].notna() & (merged["WTPH2YR"] > 0)
n_valid_hba1c = int(valid_hba1c_mask.sum())

progression = pd.DataFrame([
    {"Step": "1. Full NHANES Sample", "N": n_total, "Removed": 0, "Reason": "All survey participants"},
    {"Step": "2. Adults (Age >= 18)", "N": n_adults, "Removed": n_total - n_adults, "Reason": "Exclude pediatric (<18 years)"},
    {"Step": "3. No Known Diabetes (DIQ010=2)", "N": n_no_dm, "Removed": n_adults - n_no_dm, "Reason": "Exclude self-reported diabetes"},
    {"Step": "4. No Known Prediabetes (DIQ160=2)", "N": n_cohort_e, "Removed": n_no_dm - n_cohort_e, "Reason": "Exclude self-reported prediabetes"},
    {"Step": "5. Valid HbA1c + Phlebotomy Weight", "N": n_valid_hba1c, "Removed": n_cohort_e - n_valid_hba1c, "Reason": "Require valid LBXGH & WTPH2YR>0"},
])
display(progression)
print(f"\nCanonical Screening Population (Cohort E + Valid HbA1c): N = {n_valid_hba1c}")


                                 Step      N  Removed                             Reason
0               1. Full NHANES Sample  11933        0            All survey participants
1               2. Adults (Age >= 18)   8153     3780      Exclude pediatric (<18 years)
2     3. No Known Diabetes (DIQ010=2)   6803     1350     Exclude self-reported diabetes
3  4. No Known Prediabetes (DIQ160=2)   5907      896  Exclude self-reported prediabetes
4  5. Valid HbA1c + Phlebotomy Weight   4260     1647    Require valid LBXGH & WTPH2YR>0

Canonical Screening Population (Cohort E + Valid HbA1c): N = 4260


### INTERPRETATION
The attrition audit demonstrates that each exclusion step is clinically justified:
- **Pediatric exclusion**: Community dysglycemia screening tools apply to adult populations.
- **Known diabetes/prediabetes exclusion**: Screening aims to detect *unrecognized* disease; individuals with prior diagnoses already receive medical surveillance.
- **Valid HbA1c requirement**: Participants without laboratory results cannot serve as labeled training examples.


## 5. Initial Data Quality: Key Variable Distributions

### CONCEPT
Before feature engineering, we examine the distributions of candidate predictor variables and the reference standard (HbA1c) in the pre-filtered population.


In [7]:
# Focus on the valid HbA1c population for quality checks
pop = merged[valid_hba1c_mask].copy()

key_vars = {
    "RIDAGEYR": "Age (years)",
    "BMXBMI": "BMI (kg/m2)",
    "BMXWAIST": "Waist Circumference (cm)",
    "LBXGH": "HbA1c (%)",
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (col, label) in zip(axes, key_vars.items()):
    data = pop[col].dropna()
    ax.hist(data, bins=40, color='#4A90D9', edgecolor='white', alpha=0.85)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_ylabel('Count')
    ax.axvline(data.median(), color='#E74C3C', linestyle='--', linewidth=1.2, label=f'Median: {data.median():.1f}')
    ax.legend(fontsize=8)

plt.suptitle("Key Variable Distributions (Screening Population N=4,260)", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Missingness summary for candidate predictors
miss_cols = ["RIDAGEYR", "RIAGENDR", "BMXBMI", "BPQ020", "SMQ020", "BMXWAIST", "PAD680", "LBXGH"]
miss_data = []
for col in miss_cols:
    if col in pop.columns:
        n_miss = int(pop[col].isna().sum())
        pct_miss = n_miss / len(pop) * 100
        miss_data.append({"Variable": col, "N_Missing": n_miss, "Pct_Missing": f"{pct_miss:.2f}%"})
display(pd.DataFrame(miss_data))


   Variable  N_Missing Pct_Missing
0  RIDAGEYR          0       0.00%
1  RIAGENDR          0       0.00%
2    BMXBMI         56       1.31%
3    BPQ020          0       0.00%
4    SMQ020          2       0.05%
5  BMXWAIST        174       4.08%
6    PAD680          4       0.09%
7     LBXGH          0       0.00%


## 6. Sentinel Code Detection: PAD680

### CONCEPT
Physical activity questionnaire variable `PAD680` (sedentary minutes/day) uses sentinel codes that must be handled before modeling:
- `7777` = Refused
- `9999` = Don't Know

These are NOT valid numeric values and must be converted to `NaN` during data preparation.


In [8]:
# Detect sentinel codes in PAD680
if "PAD680" in pop.columns:
    pad_vals = pop["PAD680"].dropna()
    n_7777 = int((pad_vals == 7777).sum())
    n_9999 = int((pad_vals == 9999).sum())
    n_valid = int((pad_vals < 7777).sum())
    print(f"PAD680 (Sedentary Minutes/Day) in Screening Population:")
    print(f"  Valid responses (< 7777): {n_valid:,}")
    print(f"  Sentinel 7777 (Refused):  {n_7777}")
    print(f"  Sentinel 9999 (Don't Know): {n_9999}")
    print(f"  Total non-null: {len(pad_vals):,}")
    print(f"\n  These sentinel codes will be recoded to NaN in Data Preparation (NB03)")


PAD680 (Sedentary Minutes/Day) in Screening Population:
  Valid responses (< 7777): 4,232
  Sentinel 7777 (Refused):  1
  Sentinel 9999 (Don't Know): 23
  Total non-null: 4,256

  These sentinel codes will be recoded to NaN in Data Preparation (NB03)


## AUDIT CHECK — Data Understanding

| # | Verification Item | Status |
|:--|:------------------|:-------|
| 1 | All 8 raw XPT files present and readable | PASS |
| 2 | SEQN is unique across all files (zero duplicates) | PASS |
| 3 | Component SEQNs are subsets of DEMO_L | PASS |
| 4 | Left-join merge produces exactly N=11,933 | PASS |
| 5 | Merged microdata previewed using .head() | PASS |
| 6 | Row progression audit documented (11,933 to 4,260) | PASS |
| 7 | Sentinel codes (7777, 9999) in PAD680 identified | PASS |
| 8 | HbA1c distribution visually inspected | PASS |
